In [ ]:
%pip install fastapi uvicorn requests

In [2]:
import requests
import json
import threading
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

<span style="font-family: 'Gungsuh', '바탕체'; font-size: 2em; font-style: italic;">항목별 API 서버 코드 작성</span>

In [3]:
# 1. 항목: 기본 연결 (GET)
@app.get("/")
def home():
    return {"status": "Server is Running"}

# 2. 항목: 경로 매개변수 (Path Param)
@app.get("/user/{user_id}")
def get_user(user_id: int):
    return {"user_id": user_id, "type": "VIP"}

# 3. 항목: 데이터 전송 및 검증 (POST / Pydantic)
class Passenger(BaseModel):
    pclass: int
    sex: int
    age: float

@app.post("/predict")
def predict(item: Passenger):
    # 간단한 로직 테스트
    survival_chance = "High" if item.sex == 1 and item.pclass == 1 else "Low"
    return {"prediction": survival_chance, "received": item}

<span style="font-family: 'Gungsuh', '바탕체'; font-size: 2em; font-style: italic;">노트북 안에서 서버 가동하기</span>

In [ ]:
# 서버를 백그라운드에서 실행하는 함수
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# 이미 서버가 돌아가고 있을 수 있으므로 스레드로 실행
thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("🚀 FastAPI 서버가 http://127.0.0.1:8000 에서 가동 중입니다!")

🚀 FastAPI 서버가 http://127.0.0.1:8000 에서 가동 중입니다!


INFO:     Started server process [14768]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


<span style="font-family: 'Gungsuh', '바탕체'; font-size: 2em; font-style: italic;">항목별 클라이언트 테스트</span>

<span style="font-family: 'Gungsuh', '바탕체'; font-size: 1.3em; font-style: italic;">기본 GET 요청 테스트</span>

In [5]:
response = requests.get("http://127.0.0.1:8000/")
print(f"상태 코드: {response.status_code}")
print(f"응답 결과: {response.json()}")

INFO:     127.0.0.1:58659 - "GET / HTTP/1.1" 200 OK
상태 코드: 200
응답 결과: {'status': 'Server is Running'}


<span style="font-family: 'Gungsuh', '바탕체'; font-size: 1.3em; font-style: italic;">경로 변수(ID) 테스트</span>

In [6]:
user_id = 99
response = requests.get(f"http://127.0.0.1:8000/user/{user_id}")
print(f"결과: {response.json()}")

INFO:     127.0.0.1:58663 - "GET /user/99 HTTP/1.1" 200 OK
결과: {'user_id': 99, 'type': 'VIP'}


<span style="font-family: 'Gungsuh', '바탕체'; font-size: 1.3em; font-style: italic;">실전 데이터(POST) 전송 테스트</span>

In [7]:
# 서버로 보낼 데이터
payload = {
    "pclass": 1,
    "sex": 1,
    "age": 25.5
}

response = requests.post("http://127.0.0.1:8000/predict", json=payload)
print("--- 생존 예측 결과 ---")
print(json.dumps(response.json(), indent=4, ensure_ascii=False))

INFO:     127.0.0.1:58667 - "POST /predict HTTP/1.1" 200 OK
--- 생존 예측 결과 ---
{
    "prediction": "High",
    "received": {
        "pclass": 1,
        "sex": 1,
        "age": 25.5
    }
}


<span style="font-family: 'Gungsuh', '바탕체'; font-size: 1.3em; font-style: italic;">데이터 에러(Validation) 테스트</span>

In [8]:
bad_payload = {"pclass": "First Class", "sex": 0, "age": 30} # pclass가 int가 아님
response = requests.post("http://127.0.0.1:8000/predict", json=bad_payload)

print(f"에러 상태 코드: {response.status_code}")
print(f"에러 메시지: {response.json()['detail'][0]['msg']}")

INFO:     127.0.0.1:58669 - "POST /predict HTTP/1.1" 422 Unprocessable Content
에러 상태 코드: 422
에러 메시지: Input should be a valid integer, unable to parse string as an integer
